# Split the WhisperJAV recording for upload

This free, CPU-only Colab notebook splits `My Drive/WhisperJAV/RCT-915.m4a` into **two independently playable M4A files**, each targeted below 95,000,000 bytes (under 100 MB). It uses FFmpeg stream-copy, so it does not re-encode or reduce audio quality. The original stays untouched; the two parts are saved beside it in `WhisperJAV`.

Choose a standard CPU runtime; no GPU is needed. Run the cell and approve Drive access. The split point is adjusted automatically and the finished part sizes are checked before they are copied to Drive.

In [ ]:
#@title Split RCT-915.m4a into two parts under 100 MB
from google.colab import drive
from pathlib import Path
import shutil, subprocess, json, tempfile

drive.mount('/content/drive')
source = Path('/content/drive/MyDrive/WhisperJAV/RCT-915.m4a')
if not source.is_file():
    raise FileNotFoundError(f'Could not find {source}; check the Drive folder and filename.')

ffmpeg = shutil.which('ffmpeg')
ffprobe = shutil.which('ffprobe')
if not ffmpeg or not ffprobe:
    subprocess.run(['apt-get', 'update', '-qq'], check=True)
    subprocess.run(['apt-get', 'install', '-y', '-qq', 'ffmpeg'], check=True)
    ffmpeg = shutil.which('ffmpeg')
    ffprobe = shutil.which('ffprobe')
if not ffmpeg or not ffprobe:
    raise RuntimeError('FFmpeg/ffprobe could not be installed in this Colab runtime.')

MAX_PART_BYTES = 95_000_000  # Safety margin below the 100 MB download limit.
source_size = source.stat().st_size
print(f'Source size: {source_size:,} bytes ({source_size / 1_000_000:.1f} MB)')
if source_size >= 2 * MAX_PART_BYTES:
    raise RuntimeError('The source is too large to fit into two parts under the safety limit; use three parts instead.')

local_source = Path('/content/RCT-915.m4a')
print('Copying source to Colab local storage...')
shutil.copy2(source, local_source)
duration = float(subprocess.check_output([
    ffprobe, '-v', 'error', '-show_entries', 'format=duration',
    '-of', 'default=noprint_wrappers=1:nokey=1', str(local_source),
], text=True).strip())
if duration <= 1:
    raise RuntimeError(f'Unexpected audio duration: {duration} seconds')
print(f'Audio duration: {duration / 60:.1f} minutes. Finding a safe split point...')

work = Path('/content/whisperjav-split-parts')
work.mkdir(exist_ok=True)
def split_at(seconds):
    for old in work.glob('part*.m4a'):
        old.unlink()
    command = [
        ffmpeg, '-hide_banner', '-loglevel', 'error', '-y', '-i', str(local_source),
        '-map', '0:a:0', '-map_metadata', '0', '-c:a', 'copy',
        '-f', 'segment', '-segment_times', f'{seconds:.6f}',
        '-segment_format', 'mp4', '-reset_timestamps', '1',
        str(work / 'part%02d.m4a'),
    ]
    subprocess.run(command, check=True)
    return sorted(work.glob('part*.m4a'))

low, high = 0.0, duration
parts = []
for attempt in range(1, 18):
    split_seconds = (low + high) / 2
    parts = split_at(split_seconds)
    if len(parts) != 2:
        raise RuntimeError(f'Expected exactly two audio parts, got {len(parts)}.')
    sizes = [part.stat().st_size for part in parts]
    print(f'Check {attempt}: {sizes[0]:,} bytes + {sizes[1]:,} bytes')
    if max(sizes) <= MAX_PART_BYTES:
        break
    if sizes[0] > MAX_PART_BYTES:
        high = split_seconds  # Move the cut earlier to shrink part 1.
    else:
        low = split_seconds  # Move the cut later to shrink part 2.
else:
    raise RuntimeError('Could not find a split point that keeps both parts under 95 MB.')

destinations = [
    source.with_name(source.stem + '_part1' + source.suffix),
    source.with_name(source.stem + '_part2' + source.suffix),
]
for part, destination in zip(parts, destinations):
    shutil.copy2(part, destination)
    final_size = destination.stat().st_size
    if final_size > MAX_PART_BYTES:
        raise RuntimeError(f'{destination.name} exceeds the safety limit ({final_size:,} bytes).')
    print(f'Saved {destination.name}: {final_size:,} bytes ({final_size / 1_000_000:.1f} MB)')
print('Done. Original audio was not modified; both parts were stream-copied without re-encoding.')